# M6 Model-Diffing — Shared pt-SAE Co-Response Unit vs Best Latent (gemma-2-2b vs -it)

This notebook is a **minimal, CPU-only demo** of the M6 (model-diffing) experiment for the
Two-Track CCRG hypothesis.

**Question.** Treating SAE features as a learned knowledge representation, does a multi-latent
**co-response UNIT** (a cluster of co-activating Gemma-Scope latents) detect the RLHF *detox*
usage shift between `gemma-2-2b` (base) and `gemma-2-2b-it` (instruction-tuned) **more reliably
than the best single latent**, above a model-label shuffle null, with the shared-SAE
out-of-distribution confound bounded?

**What the full pipeline does (`method.py`).** It loads one frozen Gemma-Scope layer-12 / width-16k
JumpReLU SAE and applies it to the layer-12 residuals of *both* models, computing a BOS-excluded
max-pooled response of the co-response **unit** and of the **best single latent** for 1200 toxic
(civil_comments) and 1200 spelling-`L` corpus texts. It then runs the downstream diffing statistics
(base-vs-IT separability AUC, paired Cohen's $d_z$, a paired sign-flip null, doc-bootstrap CIs) and a
load-bearing **confound bounding** step (B2: genuine toxicity shift = toxicity departure **minus** a
spelling control floor).

**What this demo does.** The GPU model-inference step (loading two 2B models + the SAE) cannot run on
a CPU in a few minutes, so the demo loads the **precomputed per-text responses** and re-runs the
**downstream statistical analysis verbatim** — the actual methodological contribution. It reproduces
the headline result: a base-vs-IT shift *is* detectable above the null, but it is **not
concept-specific** (the spelling control shows the same departure), so the confound-controlled
genuine toxicity shift is ~0 → verdict **`clean-null-limitation`**.

No GPU, no model downloads, `$0` LLM spend.

In [1]:
# --- Install dependencies (works on Colab and locally) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# numpy / scikit-learn / matplotlib are pre-installed on Colab (do NOT reinstall there);
# locally we install them at Colab's exact versions so the environment matches.
if "google.colab" not in sys.modules:
    _pip("numpy==2.0.2", "scikit-learn==1.6.1", "matplotlib==3.10.0")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
# --- Imports (the demo runs only the statistics part, so no torch/transformers needed) ---
import json
import os
import numpy as np
from sklearn.metrics import roc_auc_score   # base-vs-IT separability AUC
import matplotlib
matplotlib.use("Agg")                        # headless backend (Colab/nbconvert safe)
import matplotlib.pyplot as plt

In [3]:
# --- Data loading: GitHub raw URL with local fallback (Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-3/experiment-5/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(data["description"])
for g in data["datasets"]:
    print(f"  dataset: {g['dataset']:<34} examples={len(g['examples'])}")
print("  reference full-run verdict:", data["reference_results"]["verdict"])

Curated per-text demo subset for the M6 model-diffing experiment. Each example carries the precomputed BOS-excluded max-pooled co-response-UNIT and best-single-latent responses on gemma-2-2b (base) vs gemma-2-2b-it (it) at layer-12 through the shared frozen Gemma-Scope pt-SAE. The demo re-runs the downstream diffing statistics (AUC, sign-flip null, doc-bootstrap CIs, B2 control-floor genuine shift) on these arrays.
  dataset: toxicity_diffing_civilcomments     examples=100
  dataset: spelling_diffing_corpus_L          examples=100
  reference full-run verdict: clean-null-limitation


## Configuration

All tunable parameters live here. The **only** parameter scaled down from the original experiment is
`N_PER_CONCEPT` — the curated demo subset holds 100 texts per concept, whereas the full run used 1200.
The resampling-iteration counts (`B_NULL`, `B_BOOT`) match the original experiment exactly, and run in
a few seconds on CPU at this data size.

In [5]:
# ----------------------------------------------------------------- CONFIG
# Demo starts small; scale up as desired (see notes). Original full-run values in comments.
N_PER_CONCEPT = 100    # texts per concept used. Full original: 1200. Demo subset max: 100.
B_NULL        = 2000   # paired sign-flip shuffle-null iterations. Original: 2000.
B_BOOT        = 2000   # doc-bootstrap CI iterations.             Original: 2000.
SEED          = 1234   # matches the original experiment seed.

# CONFIG dict so the copied-verbatim metric functions (which read CONFIG["b_null"]/["b_boot"])
# work without modification.
CONFIG = dict(b_null=B_NULL, b_boot=B_BOOT, seed=SEED)

np.random.seed(SEED)
print(f"N_PER_CONCEPT={N_PER_CONCEPT}  B_NULL={B_NULL}  B_BOOT={B_BOOT}  SEED={SEED}")

N_PER_CONCEPT=100  B_NULL=2000  B_BOOT=2000  SEED=1234


## Step 1 — Reconstruct the per-text response arrays

In the full `method.py`, these arrays come from running `gemma-2-2b` (base) and `gemma-2-2b-it` (it)
layer-12 residuals through the shared frozen Gemma-Scope SAE and max-pooling over non-BOS tokens.
Here we load them **precomputed** from the curated subset.

For each concept we get four arrays of length `N_PER_CONCEPT`:

* `b_unit`, `t_unit` — the **METHOD**: co-response **unit** response on base / it (max over member latents).
* `b_single`, `t_single` — the **BASELINE**: **best single latent** (the iter-2 anchor) on base / it.

The **toxicity** concept is primary (RLHF detox expected); the **spelling-`L`** concept is the
control (no detox expected → its measured shift is the OOD-drift artifact floor).

In [6]:
def concept_arrays(dataset_name, n_cap):
    """Pull precomputed base/it unit & single responses + doc ids for one concept."""
    g = next(g for g in data["datasets"] if g["dataset"] == dataset_name)
    ex = g["examples"][:n_cap]
    return dict(
        b_unit  =np.array([e["metadata_unit_base"]   for e in ex], dtype=np.float64),  # METHOD unit, base
        t_unit  =np.array([e["metadata_unit_it"]     for e in ex], dtype=np.float64),  # METHOD unit, it
        b_single=np.array([e["metadata_single_base"] for e in ex], dtype=np.float64),  # BASELINE single, base
        t_single=np.array([e["metadata_single_it"]   for e in ex], dtype=np.float64),  # BASELINE single, it
        doc_ids =[e["metadata_doc_id"] for e in ex],
    )

TOX_DS   = "toxicity_diffing_civilcomments"
SPELL_DS = "spelling_diffing_corpus_L"
tox   = concept_arrays(TOX_DS,   N_PER_CONCEPT)   # PRIMARY concept  (detox expected)
spell = concept_arrays(SPELL_DS, N_PER_CONCEPT)   # CONTROL concept  (no detox -> OOD-drift floor)

print(f"toxicity : n={len(tox['b_unit'])}  unique docs={len(set(tox['doc_ids']))}")
print(f"spelling : n={len(spell['b_unit'])}  unique docs={len(set(spell['doc_ids']))}")
print(f"example toxicity unit response  base={tox['b_unit'][0]:.3f}  it={tox['t_unit'][0]:.3f}")

toxicity : n=100  unique docs=100
spelling : n=100  unique docs=60
example toxicity unit response  base=5.739  it=8.337


## Step 2 — Diffing metrics (copied verbatim from `method.py`)

These are the downstream statistics, unchanged from the original script:

* `auc_base_vs_it` — separability AUC labelling base=1 / it=0 (0.5 = no shift).
* `paired_stats` — paired mean delta, Cohen's $d_z$, fraction positive.
* `shuffle_null` — paired **sign-flip** null: randomly swap (base, it) per text; departure = $|AUC-0.5|$.
* `doc_bootstrap` — resample **documents** with replacement for 95% CIs.
* `diff_metrics` — the full per-feature diffing summary (AUC + paired effect + null + CI).
* `unit_vs_single_boot` — bootstrap CI of (unit AUC − single AUC) and of the absolute-deviation difference.
* `genuine_shift_boot` — **confound B2**: genuine toxicity shift = toxicity departure − spelling floor.

In [7]:
# ============================================================================ METRICS
# (copied verbatim from method.py)

def auc_base_vs_it(b: np.ndarray, t: np.ndarray) -> float:
    """Separability AUC: label base=1, it=0, score = response value. 0.5 == no shift."""
    y = np.concatenate([np.ones(len(b)), np.zeros(len(t))])
    s = np.concatenate([b, t])
    if np.allclose(s, s[0]):
        return 0.5
    return float(roc_auc_score(y, s))


def paired_stats(b: np.ndarray, t: np.ndarray) -> dict:
    d = b - t
    sd = float(d.std()) if len(d) > 1 else 0.0
    return dict(mean_delta=float(d.mean()),
                d_z=float(d.mean() / sd) if sd > 1e-12 else 0.0,
                frac_pos=float((d > 0).mean()),
                n=int(len(d)))


def shuffle_null(b: np.ndarray, t: np.ndarray, rng, n: int) -> dict:
    """Paired sign-flip null: for each text randomly swap (b,t). Departure = |AUC-0.5| and |mean_delta|."""
    obs_auc = auc_base_vs_it(b, t)
    obs_md = float((b - t).mean())
    obs_auc_dep = abs(obs_auc - 0.5)
    obs_md_abs = abs(obs_md)
    N = len(b)
    auc_dep_null = np.empty(n, dtype=np.float64)
    md_abs_null = np.empty(n, dtype=np.float64)
    for k in range(n):
        flip = rng.random(N) < 0.5
        bb = np.where(flip, t, b)
        tt = np.where(flip, b, t)
        auc_dep_null[k] = abs(auc_base_vs_it(bb, tt) - 0.5)
        md_abs_null[k] = abs(float((bb - tt).mean()))
    return dict(
        auc=obs_auc, auc_departure=obs_auc_dep, mean_delta=obs_md,
        auc_dep_p=float((auc_dep_null >= obs_auc_dep).mean()),
        auc_dep_null_95=float(np.percentile(auc_dep_null, 95)),
        auc_pass95=bool(obs_auc_dep > np.percentile(auc_dep_null, 95)),
        md_p=float((md_abs_null >= obs_md_abs).mean()),
        md_null_95=float(np.percentile(md_abs_null, 95)),
        md_pass95=bool(obs_md_abs > np.percentile(md_abs_null, 95)),
        direction="base>it" if obs_md > 0 else "it>base",
    )


def doc_indices(doc_ids: list) -> dict:
    d = {}
    for i, doc in enumerate(doc_ids):
        d.setdefault(doc, []).append(i)
    return d


def doc_bootstrap(b: np.ndarray, t: np.ndarray, doc_map: dict, rng, n: int,
                  funcs: dict) -> dict:
    """Resample DOCS with replacement; recompute each func(b_rs,t_rs). Returns 2.5/97.5 CIs."""
    docs = list(doc_map.keys())
    nd = len(docs)
    acc = {k: np.empty(n, dtype=np.float64) for k in funcs}
    for it in range(n):
        pick = rng.integers(0, nd, size=nd)
        idx = []
        for p in pick:
            idx.extend(doc_map[docs[p]])
        idx = np.array(idx, dtype=np.int64)
        bb, tt = b[idx], t[idx]
        for k, fn in funcs.items():
            acc[k][it] = fn(bb, tt)
    out = {}
    for k, arr in acc.items():
        out[k] = dict(mean=float(arr.mean()),
                      ci_lo=float(np.percentile(arr, 2.5)),
                      ci_hi=float(np.percentile(arr, 97.5)))
    return out


def diff_metrics(b: np.ndarray, t: np.ndarray, doc_map: dict, rng) -> dict:
    """Full diffing summary for one (concept, feature): AUC, paired effect, shuffle null, doc-boot CI."""
    sh = shuffle_null(b, t, rng, CONFIG["b_null"])
    ps = paired_stats(b, t)
    boot = doc_bootstrap(b, t, doc_map, rng, CONFIG["b_boot"], funcs=dict(
        auc=lambda x, y: auc_base_vs_it(x, y),
        auc_departure=lambda x, y: abs(auc_base_vs_it(x, y) - 0.5),
        mean_delta=lambda x, y: float((x - y).mean()),
    ))
    return dict(
        auc=sh["auc"], auc_departure=sh["auc_departure"], auc_ci=boot["auc"],
        auc_departure_ci=boot["auc_departure"],
        mean_delta=ps["mean_delta"], mean_delta_ci=boot["mean_delta"],
        d_z=ps["d_z"], frac_pos=ps["frac_pos"], n=ps["n"],
        shuffle_auc_p=sh["auc_dep_p"], shuffle_auc_null_95=sh["auc_dep_null_95"],
        pass95=sh["auc_pass95"], shuffle_md_p=sh["md_p"], md_pass95=sh["md_pass95"],
        direction=sh["direction"],
    )


def unit_vs_single_boot(b_unit, t_unit, b_single, t_single, doc_map, rng, n) -> dict:
    """Bootstrap CI of (auc_unit - auc_single) and of (|auc_unit-.5| - |auc_single-.5|)."""
    docs = list(doc_map.keys())
    nd = len(docs)
    d_auc = np.empty(n, dtype=np.float64)
    d_dep = np.empty(n, dtype=np.float64)
    for it in range(n):
        pick = rng.integers(0, nd, size=nd)
        idx = []
        for p in pick:
            idx.extend(doc_map[docs[p]])
        idx = np.array(idx, dtype=np.int64)
        au = auc_base_vs_it(b_unit[idx], t_unit[idx])
        asg = auc_base_vs_it(b_single[idx], t_single[idx])
        d_auc[it] = au - asg
        d_dep[it] = abs(au - 0.5) - abs(asg - 0.5)
    au0 = auc_base_vs_it(b_unit, t_unit)
    as0 = auc_base_vs_it(b_single, t_single)
    return dict(
        auc_unit=au0, auc_single=as0, auc_diff=au0 - as0,
        auc_diff_ci=[float(np.percentile(d_auc, 2.5)), float(np.percentile(d_auc, 97.5))],
        abs_dev_diff=abs(au0 - 0.5) - abs(as0 - 0.5),
        abs_dev_diff_ci=[float(np.percentile(d_dep, 2.5)), float(np.percentile(d_dep, 97.5))],
        unit_wins=bool(np.percentile(d_dep, 2.5) > 0),
    )


def genuine_shift_boot(tox_b, tox_t, tox_doc, spell_b, spell_t, spell_doc, rng, n) -> dict:
    """B2: genuine toxicity shift = tox departure - spelling departure, resampling docs in BOTH
    concepts independently. Reported for AUC-departure and for |d_z| (scale-free effect size)."""
    tdocs = list(tox_doc.keys()); ndt = len(tdocs)
    sdocs = list(spell_doc.keys()); nds = len(sdocs)

    def dz(x, y):
        d = x - y; sd = d.std()
        return abs(float(d.mean() / sd)) if sd > 1e-12 else 0.0

    g_auc = np.empty(n, dtype=np.float64)
    g_dz = np.empty(n, dtype=np.float64)
    for it in range(n):
        ti = []
        for p in rng.integers(0, ndt, size=ndt):
            ti.extend(tox_doc[tdocs[p]])
        ti = np.array(ti, dtype=np.int64)
        si = []
        for p in rng.integers(0, nds, size=nds):
            si.extend(spell_doc[sdocs[p]])
        si = np.array(si, dtype=np.int64)
        tox_dep = abs(auc_base_vs_it(tox_b[ti], tox_t[ti]) - 0.5)
        sp_dep = abs(auc_base_vs_it(spell_b[si], spell_t[si]) - 0.5)
        g_auc[it] = tox_dep - sp_dep
        g_dz[it] = dz(tox_b[ti], tox_t[ti]) - dz(spell_b[si], spell_t[si])
    tox_dep0 = abs(auc_base_vs_it(tox_b, tox_t) - 0.5)
    sp_dep0 = abs(auc_base_vs_it(spell_b, spell_t) - 0.5)
    return dict(
        auc_departure=dict(tox=tox_dep0, spelling_floor=sp_dep0, genuine=tox_dep0 - sp_dep0,
                           ci_lo=float(np.percentile(g_auc, 2.5)), ci_hi=float(np.percentile(g_auc, 97.5)),
                           excludes_0=bool(np.percentile(g_auc, 2.5) > 0)),
        dz=dict(tox=dz(tox_b, tox_t), spelling_floor=dz(spell_b, spell_t),
                genuine=dz(tox_b, tox_t) - dz(spell_b, spell_t),
                ci_lo=float(np.percentile(g_dz, 2.5)), ci_hi=float(np.percentile(g_dz, 97.5)),
                excludes_0=bool(np.percentile(g_dz, 2.5) > 0)),
    )

print("metric functions defined")

metric functions defined


## Step 3 — Run the diffing per concept (METHOD unit vs BASELINE single latent)

For each concept we compute the full diffing summary for the co-response **unit** and the **best
single latent**. `pass95=True` means the base-vs-IT departure exceeds the 95th percentile of the
sign-flip null (i.e. a shift *is* detectable). Note `direction=it>base`: the IT model fires the
toxicity concept *more*, opposite the naive detox prediction — an early hint that this is generic
drift, not concept-specific detox.

In [8]:
tox_doc   = doc_indices(tox["doc_ids"])
spell_doc = doc_indices(spell["doc_ids"])

# rng seeds mirror the original (SEED + per-concept offset) for comparable structure.
diffing = {}
diffing["toxicity"] = {
    "unit":   diff_metrics(tox["b_unit"],   tox["t_unit"],   tox_doc,   np.random.default_rng(SEED + 11)),
    "single": diff_metrics(tox["b_single"], tox["t_single"], tox_doc,   np.random.default_rng(SEED + 12)),
}
diffing["spelling"] = {
    "unit":   diff_metrics(spell["b_unit"],   spell["t_unit"],   spell_doc, np.random.default_rng(SEED + 22)),
    "single": diff_metrics(spell["b_single"], spell["t_single"], spell_doc, np.random.default_rng(SEED + 23)),
}

for concept in ("toxicity", "spelling"):
    for feat in ("unit", "single"):
        m = diffing[concept][feat]
        print(f"{concept:9s} {feat:6s}  AUC={m['auc']:.3f}  departure={m['auc_departure']:.3f}  "
              f"dir={m['direction']:8s}  pass95={str(m['pass95']):5s}  "
              f"d_z={m['d_z']:+.3f}  null95={m['shuffle_auc_null_95']:.4f}")

toxicity  unit    AUC=0.438  departure=0.062  dir=it>base   pass95=True   d_z=-0.964  null95=0.0160
toxicity  single  AUC=0.454  departure=0.046  dir=it>base   pass95=True   d_z=-0.769  null95=0.0159
spelling  unit    AUC=0.428  departure=0.072  dir=it>base   pass95=True   d_z=-1.395  null95=0.0185
spelling  single  AUC=0.435  departure=0.065  dir=it>base   pass95=True   d_z=-0.854  null95=0.0208


## Step 4 — Does the co-response UNIT beat the best single latent?

Honest head-to-head: `unit_wins=True` only if the bootstrap CI of (unit |AUC−0.5| − single |AUC−0.5|)
excludes 0. In the **full run (n=1200) this is False** — the multi-latent unit does *not* detect the
shift more reliably than the single best latent, and the paper reports it as such.

⚠️ This comparison is the most sample-size-sensitive quantity in the experiment. At the reduced demo
scale (n=100/concept) the CI can spuriously exclude 0, so the demo may print `unit_wins=True`; both
the demo value and the `n=1200` reference are shown side by side in the results cell to make the
under-powering explicit.

In [9]:
uvs_tox = unit_vs_single_boot(
    tox["b_unit"], tox["t_unit"], tox["b_single"], tox["t_single"],
    tox_doc, np.random.default_rng(SEED + 13), CONFIG["b_boot"])

print(f"toxicity unit AUC   = {uvs_tox['auc_unit']:.4f}")
print(f"toxicity single AUC = {uvs_tox['auc_single']:.4f}")
print(f"abs-dev difference  = {uvs_tox['abs_dev_diff']:+.4f}  "
      f"CI=[{uvs_tox['abs_dev_diff_ci'][0]:+.4f}, {uvs_tox['abs_dev_diff_ci'][1]:+.4f}]")
print(f"unit_wins (CI lo > 0) = {uvs_tox['unit_wins']}")

toxicity unit AUC   = 0.4383
toxicity single AUC = 0.4544
abs-dev difference  = +0.0161  CI=[+0.0028, +0.0334]
unit_wins (CI lo > 0) = True


## Step 5 — Confound B2: genuine concept-specific shift + verdict

This is the **load-bearing** step. A shared base-trained SAE applied to IT activations is the exact
misattribution risk that crosscoder / Latent-Scaling work flags. To separate a *genuine* toxicity
shift from generic OOD/norm drift, we subtract the **spelling control floor**:

$$\text{genuine} = (\text{toxicity departure}) - (\text{spelling departure})$$

The verdict has two load-bearing conditions:

* **cond1** `detectable_shift` — toxicity unit departure exceeds the shuffle null (`pass95`).
* **cond2** `genuine_concept_specific` — the genuine-shift CI excludes 0.

If both hold → `delivered-bounded-result`; otherwise → `clean-null-limitation`. The full run gets
**cond1=True, cond2=False** (the spelling control matches the toxicity departure), so the genuine
shift is ~0 and the verdict is `clean-null-limitation`.

In [10]:
gs = genuine_shift_boot(
    tox["b_unit"],   tox["t_unit"],   tox_doc,
    spell["b_unit"], spell["t_unit"], spell_doc,
    np.random.default_rng(SEED + 999), CONFIG["b_boot"])

ad = gs["auc_departure"]
print(f"toxicity departure : {ad['tox']:.4f}")
print(f"spelling floor      : {ad['spelling_floor']:.4f}")
print(f"GENUINE shift       : {ad['genuine']:+.4f}  "
      f"CI=[{ad['ci_lo']:+.4f}, {ad['ci_hi']:+.4f}]  excludes_0={ad['excludes_0']}")

# ---- verdict (two load-bearing conditions, as in the original) ----
cond1 = bool(diffing["toxicity"]["unit"]["pass95"])   # detectable shift above shuffle null
cond2 = bool(ad["excludes_0"])                         # genuine (concept-specific) shift CI excludes 0
verdict = "delivered-bounded-result" if (cond1 and cond2) else "clean-null-limitation"

print(f"\ncond1 detectable_shift          = {cond1}")
print(f"cond2 genuine_concept_specific  = {cond2}")
print(f"DEMO VERDICT      = {verdict}")
print(f"REFERENCE VERDICT = {data['reference_results']['verdict']}  (full run, n=1200)")

toxicity departure : 0.0617
spelling floor      : 0.0722
GENUINE shift       : -0.0105  CI=[-0.0345, +0.0107]  excludes_0=False

cond1 detectable_shift          = True
cond2 genuine_concept_specific  = False
DEMO VERDICT      = clean-null-limitation
REFERENCE VERDICT = clean-null-limitation  (full run, n=1200)


## Results — demo vs reference full run

The table and plots below compare the demo (n=100/concept) against the reference full run
(n=1200/concept, from the saved `method_out.json`).

**Left plot** — base-vs-IT separability departure for each feature; the dashed line is the
shuffle-null 95th percentile (everything clears it → a shift is detectable). **Right plot** — the B2
confound: the toxicity departure and the spelling control floor are nearly identical, so the genuine
concept-specific shift (green, with bootstrap CI) sits on ~0. This is the honest
**`clean-null-limitation`** result: an infrastructure-bounded model-diffing finding where the
shared-SAE OOD confound is made explicit rather than left as future work.

In [11]:
ref = data["reference_results"]

# ---------------- console comparison table ----------------
print("=" * 76)
print(f"M6 MODEL-DIFFING  |  demo n={N_PER_CONCEPT}/concept   vs   reference n=1200/concept")
print("=" * 76)
print(f"{'metric':<34}{'demo':>14}{'reference':>14}")
table = [
    ("toxicity unit AUC",           diffing['toxicity']['unit']['auc'],            ref['diffing']['toxicity']['unit']['auc']),
    ("toxicity unit departure",     diffing['toxicity']['unit']['auc_departure'],  ref['diffing']['toxicity']['unit']['auc_departure']),
    ("toxicity single AUC",         diffing['toxicity']['single']['auc'],          ref['diffing']['toxicity']['single']['auc']),
    ("spelling(ctrl) unit departure", diffing['spelling']['unit']['auc_departure'], ref['diffing']['spelling']['unit']['auc_departure']),
    ("genuine shift (tox - floor)", gs['auc_departure']['genuine'],                ref['genuine_shift']['genuine']),
]
for name, dv, rv in table:
    print(f"{name:<34}{dv:>14.4f}{rv:>14.4f}")
print("-" * 76)
print(f"within-model sanity (reference): toxic-vs-neutral AUC  "
      f"base={ref['sanity']['base_auc_toxic_vs_neutral']:.3f}  "
      f"it={ref['sanity']['it_auc_toxic_vs_neutral']:.3f}   (>0.5 => unit IS a real toxicity detector)")
print(f"unit beats single latent? unit_wins={uvs_tox['unit_wins']}   "
      f"(reference unit_wins={ref['unit_vs_single_toxicity']['unit_wins']})")
print(f"DEMO verdict={verdict}   REFERENCE verdict={ref['verdict']}")

# ---------------- plots ----------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

# Panel 1: AUC departures, demo vs reference
labels = ["tox\nunit", "tox\nsingle", "spell\nunit", "spell\nsingle"]
demo_dep = [diffing['toxicity']['unit']['auc_departure'], diffing['toxicity']['single']['auc_departure'],
            diffing['spelling']['unit']['auc_departure'], diffing['spelling']['single']['auc_departure']]
ref_dep  = [ref['diffing']['toxicity']['unit']['auc_departure'], ref['diffing']['toxicity']['single']['auc_departure'],
            ref['diffing']['spelling']['unit']['auc_departure'], ref['diffing']['spelling']['single']['auc_departure']]
x = np.arange(len(labels)); w = 0.38
ax1.bar(x - w / 2, demo_dep, w, label=f"demo (n={N_PER_CONCEPT})", color="#4C72B0")
ax1.bar(x + w / 2, ref_dep,  w, label="reference (n=1200)",       color="#DD8452")
ax1.axhline(diffing['toxicity']['unit']['shuffle_auc_null_95'], ls="--", c="grey", lw=1,
            label="shuffle-null 95th pct")
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel("|AUC - 0.5|  (base-vs-IT departure)")
ax1.set_title("Base-vs-IT separability departure")
ax1.legend(fontsize=8)

# Panel 2: B2 genuine shift
names  = ["toxicity\ndeparture", "spelling\nfloor", "genuine\n(tox - floor)"]
vals   = [ad['tox'], ad['spelling_floor'], ad['genuine']]
colors = ["#C44E52", "#8172B3", "#55A868"]
bars = ax2.bar(names, vals, color=colors)
ax2.errorbar(2, ad['genuine'],
             yerr=[[max(ad['genuine'] - ad['ci_lo'], 0)], [max(ad['ci_hi'] - ad['genuine'], 0)]],
             fmt="none", ecolor="black", capsize=5)
ax2.axhline(0, c="black", lw=0.8)
ax2.set_ylabel("AUC departure")
ax2.set_title("B2 confound: genuine concept-specific shift ~ 0")
for b, v in zip(bars, vals):
    ax2.text(b.get_x() + b.get_width() / 2, v + 0.0015, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("m6_diffing_demo.png", dpi=110)
plt.show()
print("saved figure -> m6_diffing_demo.png")

M6 MODEL-DIFFING  |  demo n=100/concept   vs   reference n=1200/concept
metric                                      demo     reference
toxicity unit AUC                         0.4383        0.4376
toxicity unit departure                   0.0617        0.0624
toxicity single AUC                       0.4544        0.4413
spelling(ctrl) unit departure             0.0722        0.0622
genuine shift (tox - floor)              -0.0105        0.0002
----------------------------------------------------------------------------
within-model sanity (reference): toxic-vs-neutral AUC  base=0.710  it=0.729   (>0.5 => unit IS a real toxicity detector)
unit beats single latent? unit_wins=True   (reference unit_wins=False)
DEMO verdict=clean-null-limitation   REFERENCE verdict=clean-null-limitation


saved figure -> m6_diffing_demo.png
